In [ ]:
from sklearn.linear_model import Ridge

In [1]:
!git clone https://github.com/naveen-anandhan/Copper.git

Cloning into 'Copper'...
remote: Enumerating objects: 34, done.
remote: Counting objects: 100% (34/34), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 34 (delta 12), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (34/34), 11.00 MiB | 7.07 MiB/s, done.
Resolving deltas: 100% (12/12), done.


In [2]:
!pip install pandas numpy matplotlib seaborn scikit-learn xgboost

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score,accuracy_score, precision_score, recall_score


from sklearn.preprocessing import StandardScaler,LabelEncoder
from sklearn.tree import DecisionTreeClassifier
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [4]:
%cd /content/Copper

/content/Copper


In [10]:
cooper = pd.read_csv('Copper_Set.csv')
cooper.head(5)

,id,item_date,quantity tons,customer,country,status,item type,application,thickness,width,material_ref,product_ref,delivery date,selling_price
0,EC06F063-9DF0-440C-8764-0B0C05A4F6AE,20210401.0,54.151139,30156308.0,28.0,Won,W,10.0,2.00,1500.0,DEQ1 S460MC,1670798778,20210701.0,854.00
1,4E5F4B3D-DDDF-499D-AFDE-A3227EC49425,20210401.0,768.024839,30202938.0,25.0,Won,W,41.0,0.80,1210.0,0000000000000000000000000000000000104991,1668701718,20210401.0,1047.00
2,E140FF1B-2407-4C02-A0DD-780A093B1158,20210401.0,386.127949,30153963.0,30.0,Won,WI,28.0,0.38,952.0,S0380700,628377,20210101.0,644.33
3,F8D507A0-9C62-4EFE-831E-33E1DA53BB50,20210401.0,202.411065,30349574.0,32.0,Won,S,59.0,2.30,1317.0,DX51D+ZM310MAO 2.3X1317,1668701718,20210101.0,768.00
4,4E1C4E78-152B-430A-8094-ADD889C9D0AD,20210401.0,785.526262,30211560.0,28.0,Won,W,10.0,4.00,2000.0,2_S275JR+AR-CL1,640665,20210301.0,577.00


In [11]:
def fill_na_values(df):
    median_col = ['quantity_tons', 'customer', 'country', 'application', 'width', 'selling_price', 'thickness']
    for col in median_col:
        df[col] = df[col].apply(lambda x: np.nan if x <= 0 else x)  # Replace non-positive values with NaN
        df[col] = df[col].fillna(df[col].median())

    mode_col = ['item_date', 'delivery_date', 'status']
    for col in mode_col:
        df[col] = df[col].fillna(df[col].mode().iloc[0])
    return df

def cleaning_df(df):

    cleaning = (cooper
        .rename(columns={'quantity tons':'quantity_tons', 'item type':'item_type', 'delivery date':'delivery_date'})  #naming the column
        .assign(
            item_date=lambda x: pd.to_datetime(x['item_date'], format='%Y%m%d', errors='coerce'),
            delivery_date=lambda x: pd.to_datetime(x['delivery_date'], format='%Y%m%d', errors='coerce'),  #fixing the frature to datetime dtypes
            material_ref=lambda x: x['material_ref']
                        .str.strip('0')
                        .str.replace(r'[_+-]', '', regex=True)
                        .fillna('Unknown')
                        .astype('category'),
            quantity_tons=lambda x: pd.to_numeric(x['quantity_tons'], errors='coerce').apply(lambda y: np.nan if y <= 0 else y),
            customer=lambda x: x['customer'],
            country=lambda x: x['country'],
            application=lambda x: x['application'],
            selling_price=lambda x: pd.to_numeric(x['selling_price'], errors='coerce').apply(lambda y: np.nan if y <= 0 else y))
        .dropna()         #droping the null values becouse they are less the one percen
        .pipe(fill_na_values)
        .dropna()
        .astype({'country':'int32'})
        .reset_index()
        .drop(columns=['id','material_ref','index'])
        .sample(n=50000, random_state=42)   # Random 50,000 rows
        .reset_index(drop=True)             # Optional: reset row numbers
)
    return cleaning

df = cleaning_df(cooper)

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   item_date      50000 non-null  datetime64[ns]
 1   quantity_tons  50000 non-null  float64       
 2   customer       50000 non-null  float64       
 3   country        50000 non-null  int32         
 4   status         50000 non-null  object        
 5   item_type      50000 non-null  object        
 6   application    50000 non-null  float64       
 7   thickness      50000 non-null  float64       
 8   width          50000 non-null  float64       
 9   product_ref    50000 non-null  int64         
 10  delivery_date  50000 non-null  datetime64[ns]
 11  selling_price  50000 non-null  float64       
dtypes: datetime64[ns](2), float64(6), int32(1), int64(1), object(2)
memory usage: 4.4+ MB


In [17]:
EDA = df.copy(True)
EDA.head(6)

,item_date,quantity_tons,customer,country,status,item_type,application,thickness,width,product_ref,delivery_date,selling_price
0,2020-07-21,282.085929,30165489.0,27,Not lost for AM,S,41.0,1.20,1500.0,611993,2020-10-01,933.0
1,2021-02-04,13.255074,30271641.0,84,Lost,W,79.0,0.60,1250.0,164141591,2021-07-01,1058.0
2,2020-07-07,6.204865,30161088.0,78,Won,W,10.0,1.20,1000.0,628377,2020-08-01,526.0
3,2021-03-02,131.362415,30205190.0,84,Won,W,10.0,0.70,1500.0,164141591,2021-04-01,1043.0
4,2021-01-26,4505.880148,30403047.0,28,Won,PL,10.0,3.18,1478.0,1670798778,2021-01-01,547.0
5,2021-01-28,35.246275,30336052.0,78,Won,W,10.0,1.00,1250.0,628377,2021-04-01,849.0


In [18]:
outlier_columns = ['quantity_tons', 'thickness', 'width', 'selling_price']
before_stats = EDA[outlier_columns].describe().T
before_stats

,count,mean,std,min,25%,50%,75%,max
quantity_tons,50000.0,1098.632829,223606.903910,0.00195,11.122744,30.505581,67.669824,50000000.00
thickness,50000.0,2.529882,2.795919,0.18000,0.700000,1.500000,3.000000,25.00
width,50000.0,1295.494059,261.901939,20.50000,1179.000000,1250.000000,1500.000000,2300.00
selling_price,50000.0,818.221281,427.613662,11.00000,669.000000,812.000000,953.000000,81236.14


In [19]:
for column in outlier_columns:
    Q1 = EDA[column].quantile(0.25)
    Q3 = EDA[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    EDA[column] = EDA[column].clip(lower_bound, upper_bound)

# Calculate summary statistics after removing outliers
after_stats = EDA[outlier_columns].describe().T

after_stats

,count,mean,std,min,25%,50%,75%,max
quantity_tons,50000.0,49.412117,48.213563,0.00195,11.122744,30.505581,67.669824,152.490444
thickness,50000.0,2.235372,1.884169,0.18000,0.700000,1.500000,3.000000,6.450000
width,50000.0,1297.096801,250.003349,697.50000,1179.000000,1250.000000,1500.000000,1981.500000
selling_price,50000.0,819.050607,209.166278,243.00000,669.000000,812.000000,953.000000,1379.000000


In [21]:
from scipy import stats

columns_to_transform = ['quantity_tons','thickness','selling_price']

for col in columns_to_transform:
    EDA[f'{col}_yeojohnson'], _ = stats.yeojohnson(EDA[col])
    print(f"Skewness of {col}_yeojohnson: {EDA[f'{col}_yeojohnson'].skew()}")

Skewness of quantity_tons_yeojohnson: -0.04908332093580844
Skewness of thickness_yeojohnson: 0.1361328856156328
Skewness of selling_price_yeojohnson: 0.03757866010573251


In [22]:
data=EDA.copy(True)
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 15 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   item_date                 50000 non-null  datetime64[ns]
 1   quantity_tons             50000 non-null  float64       
 2   customer                  50000 non-null  float64       
 3   country                   50000 non-null  int32         
 4   status                    50000 non-null  object        
 5   item_type                 50000 non-null  object        
 6   application               50000 non-null  float64       
 7   thickness                 50000 non-null  float64       
 8   width                     50000 non-null  float64       
 9   product_ref               50000 non-null  int64         
 10  delivery_date             50000 non-null  datetime64[ns]
 11  selling_price             50000 non-null  float64       
 12  quantity_tons_yeoj

In [23]:
data['delivery_time_taken']=(data['item_date']-data['delivery_date']).abs().dt.days

In [24]:
data.drop(columns=['quantity_tons','thickness','selling_price','delivery_date','item_date'], inplace=True)

data.head()

,customer,country,status,item_type,application,width,product_ref,quantity_tons_yeojohnson,thickness_yeojohnson,selling_price_yeojohnson,delivery_time_taken
0,30165489.0,27,Not lost for AM,S,41.0,1500.0,611993,7.680940,0.633513,501.294015,72
1,30271641.0,84,Lost,W,79.0,1250.0,164141591,3.299754,0.411680,560.911648,147
2,30161088.0,78,Won,W,10.0,1000.0,628377,2.316455,0.633513,300.305140,25
3,30205190.0,84,Won,W,10.0,1500.0,164141591,7.357418,0.457109,553.799180,30
4,30403047.0,28,Won,PL,10.0,1478.0,1670798778,7.680940,0.973920,311.005674,25


In [26]:
from sklearn.preprocessing import OrdinalEncoder

encode = OrdinalEncoder()
data["country"] = encode.fit_transform(data[["country"]])
transformed_country = data["country"].unique()

data['status'] = encode.fit_transform(data[['status']])
transformed_status = data['status'].unique()

data["item_type"] = encode.fit_transform(data[["item_type"]])
transformed_item = data["item_type"].unique()

In [27]:
import pickle

file_path='country.pkl'
file_path2='status.pkl'
file_path3='item_type.pkl'

with open(file_path, 'wb') as file:
    pickle.dump(transformed_country, file)
with open(file_path2, 'wb') as file:
    pickle.dump(transformed_status, file)
with open(file_path3, 'wb') as file:
    pickle.dump(transformed_item, file)

print(f'Pickle file created: {file_path}, {file_path2},{file_path3}')

Pickle file created: country.pkl, status.pkl,item_type.pkl


In [28]:

from sklearn import preprocessing

X = data[['quantity_tons_yeojohnson','thickness_yeojohnson','width','status','country','item_type','application','product_ref','delivery_time_taken']].values
y = data[['selling_price_yeojohnson']].values

scalled_data = StandardScaler().fit(X)
X = scalled_data.transform(X)

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [29]:
with open('scaling.pkl', 'wb') as file:
    pickle.dump(scalled_data, file)

In [30]:
models = {"ExtraTreeRegressor" : ExtraTreesRegressor(),
          'XGBRegressor' : XGBRegressor(),
          'DecisionTreeRegressor': DecisionTreeRegressor()
}
n = X_test.shape[0]  # number of samples
p = X_test.shape[1]  # number of predictors

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    MSE = mean_squared_error(y_test, y_pred)
    MAE = mean_absolute_error(y_test, y_pred)
    R2 = r2_score(y_test, y_pred)
    Adjusted_R2 = 1 - ((1 - R2) * (n - 1) / (n - p - 1))

    print("+----------------------------------+")
    print(f"   -- {list(models.keys())[i]} --    ")
    print("+----------------------------------+")
    print("| Model performance on testing set |")
    print("+----------------------------------+")
    print("|         MSE :{:.4f}           |".format(MSE))
    print("|         MAE :{:.4f}             |".format(MAE))
    print("|         R2  :{:.4f}              |".format(R2))
    print("| Adjusted R2 : {:.4f}             |".format(Adjusted_R2))
    print("+----------------------------------+")
    print(f'='*36)

+----------------------------------+
   -- ExtraTreeRegressor --    
+----------------------------------+
| Model performance on testing set |
+----------------------------------+
|         MSE :1729.8704           |
|         MAE :28.1791             |
|         R2  :0.8331              |
| Adjusted R2 : 0.8330             |
+----------------------------------+
+----------------------------------+
   -- XGBRegressor --    
+----------------------------------+
| Model performance on testing set |
+----------------------------------+
|         MSE :2132.5212           |
|         MAE :34.8317             |
|         R2  :0.7942              |
| Adjusted R2 : 0.7941             |
+----------------------------------+
+----------------------------------+
   -- DecisionTreeRegressor --    
+----------------------------------+
| Model performance on testing set |
+----------------------------------+
|         MSE :2980.3717           |
|         MAE :33.4232             |
|         R2  :0.71

In [31]:

x = data[[ 'quantity_tons_yeojohnson','thickness_yeojohnson','width','selling_price_yeojohnson','country','item_type','application','product_ref','delivery_time_taken']].values
Y = data[['status']].values

scaler_classify = StandardScaler().fit(x)
x = scaler_classify.transform(x)

# Split the dataset into training and testing sets
x_train, x_test, Y_train, Y_test = train_test_split(x, Y, test_size=0.3, random_state=20)

In [32]:
with open('scaling_classify.pkl', 'wb') as file:
    pickle.dump(scaler_classify, file)

In [33]:

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [34]:
models = {"RandomForestClassifier" : RandomForestClassifier(),             # Random Forest Classifier
          'XGBClassifier' : xgb.XGBClassifier(),
          'DecisionTreeClassifier': DecisionTreeClassifier()
}

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(x_train, Y_train)

    # Predictions
    Y_train_pred = model.predict(x_train)
    Y_test_pred = model.predict(x_test)

    # Training metrics
    train_accuracy = accuracy_score(Y_train, Y_train_pred)
    train_precision = precision_score(Y_train, Y_train_pred, average='weighted')
    train_recall = recall_score(Y_train, Y_train_pred, average='weighted')
    train_f1 = f1_score(Y_train, Y_train_pred, average='weighted')

    # Testing metrics
    test_accuracy = accuracy_score(Y_test, Y_test_pred)
    test_precision = precision_score(Y_test, Y_test_pred, average='weighted')
    test_recall = recall_score(Y_test, Y_test_pred, average='weighted')
    test_f1 = f1_score(Y_test, Y_test_pred, average='weighted')

    print("+----------------------------------+")
    print(f"   -- {list(models.keys())[i]} --    ")
    print("+----------------------------------+")
    print("| Model performance on training set |")
    print("+----------------------------------+")
    print(f"|   Accuracy: {train_accuracy:.4f}               |")
    print(f"|   Precision: {train_precision:.4f}              |")
    print(f"|   Recall: {train_recall:.4f}                 |")
    print(f"|   F1 Score: {train_f1:.4f}               |")
    print("+----------------------------------+")
    print("| Model performance on testing set |")
    print("+----------------------------------+")
    print(f"|   Accuracy: {test_accuracy:.4f}               |")
    print(f"|   Precision: {test_precision:.4f}              |")
    print(f"|   Recall: {test_recall:.4f}                 |")
    print(f"|   F1 Score: {test_f1:.4f}               |")
    print(f'='*50)

+----------------------------------+
   -- RandomForestClassifier --    
+----------------------------------+
| Model performance on training set |
+----------------------------------+
|   Accuracy: 1.0000               |
|   Precision: 1.0000              |
|   Recall: 1.0000                 |
|   F1 Score: 1.0000               |
+----------------------------------+
| Model performance on testing set |
+----------------------------------+
|   Accuracy: 0.8531               |
|   Precision: 0.8490              |
|   Recall: 0.8531                 |
|   F1 Score: 0.8462               |
+----------------------------------+
   -- XGBClassifier --    
+----------------------------------+
| Model performance on training set |
+----------------------------------+
|   Accuracy: 0.8869               |
|   Precision: 0.8861              |
|   Recall: 0.8869                 |
|   F1 Score: 0.8834               |
+----------------------------------+
| Model performance on testing set |
+---------